In [1]:
import pandas as pd
import numpy as np

In [3]:
prev = pd.read_csv("D:/AI/HomeCredit/Processed_data/prev_final.csv")
bureau = pd.read_csv("D:/AI/HomeCredit/Processed_data/bureau_final.csv")
posinstcc = pd.read_csv("D:/AI/HomeCredit/Processed_data/pos_inst_cc_final.csv")

In [4]:
app_train = pd.read_csv("D:/AI/HomeCredit/Dataset/application_train.csv")
app_test = pd.read_csv("D:/AI/HomeCredit/Dataset/application_test.csv")

In [17]:
posinstcc = posinstcc.astype('float32')
prev = prev.astype('float32')
bureau = bureau.astype('float32')

In [18]:
print(X.dtypes.value_counts())

float64    363
bool       124
int64       39
Name: count, dtype: int64


In [19]:
def preprocess_application(df):
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'])
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'])
    df['CREDIT_TERM'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'])
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365
    df = pd.get_dummies(df, drop_first=True)
    df.fillna(-999, inplace=True)
    return df

In [20]:
app_train_processed = preprocess_application(app_train.copy())
app_test_processed  = preprocess_application(app_test.copy())

In [21]:
app_train_processed, app_test_processed = app_train_processed.align(
    app_test_processed,
    join='left',
    axis=1,
    fill_value=0
)

In [22]:
y = app_train_processed['TARGET']
X = app_train_processed.drop('TARGET', axis=1)
X_test = app_test_processed.drop('TARGET', axis=1)

In [23]:
X = X.merge(bureau, on='SK_ID_CURR', how='left')
X = X.merge(prev, on='SK_ID_CURR', how='left')
X = X.merge(posinstcc, on='SK_ID_CURR', how='left')
X_test = X_test.merge(bureau, on='SK_ID_CURR', how='left')
X_test = X_test.merge(prev, on='SK_ID_CURR', how='left')
X_test = X_test.merge(posinstcc, on='SK_ID_CURR', how='left')

In [33]:
import re
def clean_column_names(df):
    df.columns = [
        re.sub(r'_+', '_', re.sub(r'[^A-Za-z0-9_]+', '_', col)).strip('_')
        for col in df.columns
    ]
    return df

In [34]:
X = clean_column_names(X)
X_test = clean_column_names(X_test)

In [35]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [36]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [38]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(100)]
)

[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.439323 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 70788
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 516
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080794 -> initscore=-2.431606
[LightGBM] [Info] Start training from score -2.431606
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[484]	valid_0's auc: 0.781469	valid_0's binary_logloss: 0.238357


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, n_estimators=1000,
               random_state=42, subsample=0.8)

In [39]:
y_pred = model.predict_proba(X_val)[:, 1]

In [40]:
print("AUC:", roc_auc_score(y_val, y_pred))

AUC: 0.7814694464647729


In [41]:
test_pred = model.predict_proba(X_test)[:, 1]

In [43]:
submission = pd.DataFrame({
    'SK_ID_CURR': app_test['SK_ID_CURR'],
    'TARGET': test_pred
})

submission.to_csv('D:/AI/HomeCredit/Submission/submission.csv', index=False)